# Crop Fitness Tracker — Weather & CWR Processing

**What this notebook does:**
- Step 1 — Load and inspect all three input CSVs
- Step 2 — Compute daily ET₀ (FAO-56 Penman-Monteith)
- Step 3 — Assign daily Kc per crop from growth stages
- Step 4 — Compute daily CWR per crop
- Step 5 — Build and export master CSV

**Input files:**
```
nasa_power_weather.csv   — daily weather Apr 12 – Jul 30 2025
crop_stages.csv          — planting dates, stage names, start/end dates
kc_values.csv            — Kc ini, Kc mid, Kc end per crop
```

**Output file:**
```
crop_fitness_master.csv  — one row per day, all signals ready for dashboard
```

## Setup — Install and Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

---
## Step 1 — Load and Inspect Input Files

In [ ]:
# ── Load NASA POWER weather data ──────────────────────────────
weather = pd.read_csv('nasa_power_weather.csv')
weather.columns = weather.columns.str.strip()
weather['date'] = pd.to_datetime(weather['date'])
weather = weather.sort_values('date').reset_index(drop=True)

print(f'Weather records: {len(weather)} days')
print(f'Date range: {weather.date.min().date()} → {weather.date.max().date()}')
print(f'Columns: {list(weather.columns)}')
weather.head()

In [ ]:
# ── Check for missing values in weather ───────────────────────
print('Missing values per column:')
print(weather.isnull().sum())

# NASA POWER uses -999 as a fill value for missing data
# Replace with NaN
weather.replace(-999, np.nan, inplace=True)
weather.replace(-99, np.nan, inplace=True)

print('\nAfter replacing fill values:')
print(weather.isnull().sum())

In [ ]:
# ── Load crop stage data ──────────────────────────────────────
stages = pd.read_csv('crop_stages.csv')
stages.columns = stages.columns.str.strip().str.lower()

# Standardise crop names to lowercase
stages['crop'] = stages['crop'].str.strip().str.lower()

# Parse dates
stages['stage_start_date'] = pd.to_datetime(stages['stage_start_date'])
stages['stage_end_date']   = pd.to_datetime(stages['stage_end_date'])

# Filter to our three crops only
CROPS = ['cotton', 'corn', 'soybean']
stages = stages[stages['crop'].isin(CROPS)].reset_index(drop=True)

print(f'Stage records: {len(stages)}')
stages

In [ ]:
# ── Load Kc values ────────────────────────────────────────────
kc_raw = pd.read_csv('kc_values.csv')
kc_raw.columns = kc_raw.columns.str.strip()
kc_raw['Crop'] = kc_raw['Crop'].str.strip().str.lower()

print('Kc anchor values (FAO-56):')
print(kc_raw)
kc_raw

In [ ]:
# ── Build full Kc table per stage via linear interpolation ────
# FAO-56 has 3 anchor points: Kc_ini, Kc_mid, Kc_end
# We have 5 stages: Establishment, Vegetative, Flowering,
#                   Yield Formation, Ripening
#
# Mapping:
#   Establishment  → Kc_ini
#   Vegetative     → interpolate halfway between Kc_ini and Kc_mid
#   Flowering      → Kc_mid
#   Yield Formation→ Kc_mid  (sustained peak)
#   Ripening       → Kc_end

kc_table = {}

for _, row in kc_raw.iterrows():
    crop    = row['Crop']
    kc_ini  = row['Kc ini']
    kc_mid  = row['Kc mid']
    kc_end  = row['Kc end']
    kc_veg  = round(kc_ini + (kc_mid - kc_ini) * 0.5, 3)  # linear midpoint

    kc_table[crop] = {
        'Establishment'  : kc_ini,
        'Vegetative'     : kc_veg,
        'Flowering'      : kc_mid,
        'Yield formation': kc_mid,
        'Ripening'       : kc_end
    }

print('Interpolated Kc table:')
kc_df = pd.DataFrame(kc_table).T
kc_df.index.name = 'crop'
print(kc_df)
kc_df

---
## Step 2 — Compute Daily ET₀ (FAO-56 Penman-Monteith)

**Formula:**
$$ET_0 = \frac{0.408 \Delta (R_n - G) + \gamma \frac{900}{T+273} u_2 (e_s - e_a)}{\Delta + \gamma (1 + 0.34 u_2)}$$

Where:
- $R_n$ = net radiation (MJ/m²/day)
- $G$ = soil heat flux (≈ 0 for daily)
- $T$ = mean temperature (°C)
- $u_2$ = wind speed at 2m (m/s)
- $e_s$ = saturation vapour pressure (kPa)
- $e_a$ = actual vapour pressure (kPa)
- $\Delta$ = slope of saturation vapour pressure curve
- $\gamma$ = psychrometric constant

In [ ]:
def compute_et0(row, elevation_m=60):
    """
    FAO-56 Penman-Monteith reference ET₀
    elevation_m: Macon County, AL is approximately 60m above sea level
    Returns ET₀ in mm/day
    """
    T     = row['T2M']               # mean temperature °C
    T_max = row['T2M_MAX']           # max temperature °C
    T_min = row['T2M_MIN']           # min temperature °C
    RH    = row['RH2M']              # relative humidity %
    u2    = row['WS2M']              # wind speed at 2m m/s
    Rs    = row['ALLSKY_SFC_SW_DWN'] # solar radiation MJ/m²/day

    # Return NaN if any required variable is missing
    if any(pd.isna([T, T_max, T_min, RH, u2, Rs])):
        return np.nan

    # ── Atmospheric pressure from elevation ───────────────────
    P = 101.3 * ((293 - 0.0065 * elevation_m) / 293) ** 5.26  # kPa

    # ── Psychrometric constant ────────────────────────────────
    gamma = 0.000665 * P  # kPa/°C

    # ── Saturation vapour pressure ────────────────────────────
    es_Tmax = 0.6108 * np.exp(17.27 * T_max / (T_max + 237.3))
    es_Tmin = 0.6108 * np.exp(17.27 * T_min / (T_min + 237.3))
    es      = (es_Tmax + es_Tmin) / 2  # kPa

    # ── Actual vapour pressure from RH ────────────────────────
    ea = es * (RH / 100)  # kPa

    # ── Slope of saturation vapour pressure curve ─────────────
    delta = 4098 * (0.6108 * np.exp(17.27 * T / (T + 237.3))) / (T + 237.3) ** 2

    # ── Net radiation ─────────────────────────────────────────
    # Net shortwave radiation
    Rns = (1 - 0.23) * Rs  # albedo = 0.23 for reference grass surface

    # Net longwave radiation (simplified)
    sigma = 4.903e-9  # Stefan-Boltzmann MJ/m²/day/K⁴
    T_max_K = T_max + 273.16
    T_min_K = T_min + 273.16
    Rnl = sigma * ((T_max_K**4 + T_min_K**4) / 2) * \
          (0.34 - 0.14 * np.sqrt(ea)) * \
          (1.35 * (Rs / (0.75 * Rs + 0.1)) - 0.35)

    Rn = Rns - Rnl  # net radiation MJ/m²/day
    G  = 0          # soil heat flux ≈ 0 for daily timestep

    # ── FAO-56 Penman-Monteith ────────────────────────────────
    numerator   = 0.408 * delta * (Rn - G) + gamma * (900 / (T + 273)) * u2 * (es - ea)
    denominator = delta + gamma * (1 + 0.34 * u2)

    et0 = numerator / denominator
    return round(max(et0, 0), 3)  # ET₀ cannot be negative


# Apply to every row
weather['ET0'] = weather.apply(compute_et0, axis=1)

print('ET₀ computed successfully')
print(f'Mean ET₀: {weather["ET0"].mean():.2f} mm/day')
print(f'Min ET₀:  {weather["ET0"].min():.2f} mm/day')
print(f'Max ET₀:  {weather["ET0"].max():.2f} mm/day')
weather[['date','T2M','T2M_MAX','T2M_MIN','RH2M','WS2M','ALLSKY_SFC_SW_DWN','ET0']].head(10)

In [ ]:
# ── Plot ET₀ across the season ────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(weather['date'], weather['ET0'], color='#E74C3C', linewidth=1.5, label='ET₀ mm/day')
ax.fill_between(weather['date'], weather['ET0'], alpha=0.15, color='#E74C3C')
ax.set_title('Daily Reference ET₀ — Macon County 2025', fontsize=13)
ax.set_ylabel('ET₀ (mm/day)')
ax.set_xlabel('Date')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Expected: ET₀ rises from ~3 mm/day in April to ~6-8 mm/day in July')

---
## Step 3 — Assign Daily Kc Per Crop From Growth Stages

In [ ]:
def get_stage_and_kc(date, crop):
    """
    For a given date and crop returns the growth stage name
    and corresponding Kc value.
    Returns (None, None) if date is outside the crop season.
    """
    crop_stages = stages[stages['crop'] == crop]

    for _, stage_row in crop_stages.iterrows():
        if stage_row['stage_start_date'] <= date <= stage_row['stage_end_date']:
            stage_name = stage_row['stage']
            # Normalise stage name to match kc_table keys
            stage_key  = stage_name.strip().title()

            # Handle 'Yield Formation' vs 'Yield formation'
            if 'yield' in stage_name.lower():
                stage_key = 'Yield formation'

            kc = kc_table[crop].get(stage_key, np.nan)
            return stage_name, kc

    return None, np.nan  # date outside season


# Assign stage and Kc for each crop on each date
for crop in CROPS:
    stage_col = f'{crop}_stage'
    kc_col    = f'{crop}_kc'
    results   = weather['date'].apply(lambda d: get_stage_and_kc(d, crop))
    weather[stage_col] = results.apply(lambda x: x[0])
    weather[kc_col]    = results.apply(lambda x: x[1])

print('Stage and Kc assigned for all crops')
weather[['date','cotton_stage','cotton_kc','corn_stage','corn_kc','soybean_stage','soybean_kc']].head(20)

In [ ]:
# ── Verify stage transitions look correct ─────────────────────
for crop in CROPS:
    print(f'\n{crop.upper()} — stage transitions:')
    stage_col = f'{crop}_stage'
    transitions = weather[['date', stage_col]].dropna()
    # Show first date of each stage
    first_dates = transitions.groupby(stage_col)['date'].min().sort_values()
    print(first_dates.to_string())

---
## Step 4 — Compute Daily CWR Per Crop

$$CWR = ET_0 \times Kc$$

Units: mm/day

In [ ]:
for crop in CROPS:
    kc_col  = f'{crop}_kc'
    cwr_col = f'{crop}_cwr'
    weather[cwr_col] = (weather['ET0'] * weather[kc_col]).round(3)

print('CWR computed for all crops')
weather[['date','ET0',
         'cotton_kc','cotton_cwr',
         'corn_kc','corn_cwr',
         'soybean_kc','soybean_cwr']].head(20)

In [ ]:
# ── Plot CWR per crop across the season ───────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

colors = {'cotton': '#8B4513', 'corn': '#228B22', 'soybean': '#4169E1'}

for crop in CROPS:
    cwr_col = f'{crop}_cwr'
    ax.plot(weather['date'], weather[cwr_col],
            label=f'{crop.title()} CWR',
            color=colors[crop], linewidth=1.5)

ax.plot(weather['date'], weather['ET0'],
        label='ET₀ (reference)', color='gray',
        linewidth=1, linestyle='--', alpha=0.7)

ax.set_title('Daily Crop Water Requirement (CWR) — Macon County 2025', fontsize=13)
ax.set_ylabel('CWR (mm/day)')
ax.set_xlabel('Date')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Summary stats
for crop in CROPS:
    cwr_col = f'{crop}_cwr'
    vals = weather[cwr_col].dropna()
    print(f'{crop.title():10s} CWR — mean: {vals.mean():.2f}  max: {vals.max():.2f}  mm/day')

---
## Step 5 — Build and Export Master CSV

In [ ]:
# ── Select and order final columns ────────────────────────────
master = weather[[
    # Date
    'date',

    # Raw weather
    'T2M', 'T2M_MAX', 'T2M_MIN',
    'RH2M', 'WS2M',
    'ALLSKY_SFC_SW_DWN',
    'PRECTOTCORR',

    # Computed signals
    'ET0',

    # Cotton
    'cotton_stage', 'cotton_kc', 'cotton_cwr',

    # Corn
    'corn_stage', 'corn_kc', 'corn_cwr',

    # Soybean
    'soybean_stage', 'soybean_kc', 'soybean_cwr'

]].copy()

# Rename for clarity
master.rename(columns={
    'ALLSKY_SFC_SW_DWN': 'solar_rad_MJ',
    'PRECTOTCORR'      : 'precip_mm',
    'T2M'              : 'temp_mean_C',
    'T2M_MAX'          : 'temp_max_C',
    'T2M_MIN'          : 'temp_min_C',
    'RH2M'             : 'rh_pct',
    'WS2M'             : 'wind_ms'
}, inplace=True)

print(f'Master CSV shape: {master.shape}')
print(f'Columns: {list(master.columns)}')
master.head(10)

In [ ]:
# ── Final data quality check ──────────────────────────────────
print('=== FINAL DATA QUALITY CHECK ===')
print(f'Total rows: {len(master)}')
print(f'Date range: {master.date.min().date()} → {master.date.max().date()}')
print()
print('Missing values:')
print(master.isnull().sum())
print()
print('Value ranges:')
print(master.describe().round(3))

In [ ]:
# ── Export ────────────────────────────────────────────────────
OUTPUT_PATH = 'crop_fitness_master.csv'
master.to_csv(OUTPUT_PATH, index=False)

print(f'Saved: {OUTPUT_PATH}')
print(f'Shape: {master.shape}')
print()
print('Columns in output:')
for col in master.columns:
    print(f'  {col}')

In [ ]:
# ── Final summary plot ────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)

# ET₀
axes[0].plot(master['date'], master['ET0'], color='#E74C3C', linewidth=1.5)
axes[0].fill_between(master['date'], master['ET0'], alpha=0.15, color='#E74C3C')
axes[0].set_ylabel('ET₀ (mm/day)')
axes[0].set_title('Reference ET₀')
axes[0].grid(alpha=0.3)

# CWR per crop
colors = {'cotton': '#8B4513', 'corn': '#228B22', 'soybean': '#4169E1'}
for crop in CROPS:
    axes[1].plot(master['date'], master[f'{crop}_cwr'],
                 label=crop.title(), color=colors[crop], linewidth=1.5)
axes[1].set_ylabel('CWR (mm/day)')
axes[1].set_title('Crop Water Requirement per Crop')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Precipitation
axes[2].bar(master['date'], master['precip_mm'], color='#3498DB', alpha=0.7, label='Rainfall')
axes[2].set_ylabel('Precipitation (mm)')
axes[2].set_title('Daily Precipitation')
axes[2].legend()
axes[2].grid(alpha=0.3)

fig.suptitle('Crop Fitness Tracker — Macon County 2025 Season Summary', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('season_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved: season_summary.png')

---
## Done

**Output files generated:**
```
crop_fitness_master.csv   — master signal file for dashboard
season_summary.png        — season overview plot
```

**Columns in crop_fitness_master.csv:**

| Column | Description | Units |
|---|---|---|
| date | Calendar date | YYYY-MM-DD |
| temp_mean_C | Mean air temperature | °C |
| temp_max_C | Max air temperature | °C |
| temp_min_C | Min air temperature | °C |
| rh_pct | Relative humidity | % |
| wind_ms | Wind speed at 2m | m/s |
| solar_rad_MJ | Solar radiation | MJ/m²/day |
| precip_mm | Precipitation | mm/day |
| ET0 | Reference evapotranspiration | mm/day |
| cotton_stage | Growth stage name | — |
| cotton_kc | Crop coefficient | — |
| cotton_cwr | Crop water requirement | mm/day |
| corn_stage | Growth stage name | — |
| corn_kc | Crop coefficient | — |
| corn_cwr | Crop water requirement | mm/day |
| soybean_stage | Growth stage name | — |
| soybean_kc | Crop coefficient | — |
| soybean_cwr | Crop water requirement | mm/day |

**Next step:** Use `crop_fitness_master.csv` to generate synthetic ML training data.